# ANIMA-MECHANICUS — Fregate U-ALPHA : L'Auspex Cogitateur
> *L'Ame du Mouvement, Forgee par la Machine*

**Pipeline** : `.mp4 (humain reel)` → `[Gemini 2.0 Flash]` → `[WHAM SMPL]` → `[Retargeting R15]` → `.npz`

**Instructions** :
1. Cellule 1 — Installer les dependances (WHAM, Torch, Detectron2, Gemini)
2. Cellule 2 — Configurer la cle Gemini et les parametres pipeline
3. Cellule 3 — Uploader la video `.mp4` source
4. Cellule 4 — Lancer l'analyse Gemini et afficher le rapport de segments
5. Cellule 5 — Valider ou ajuster les segments a extraire
6. Cellule 6 — Lancer WHAM → SMPL→R15 → exporter `.npz`
7. Cellule 7 — Telecharger les `.npz` (input de la Fregate U-GAMMA)

**Runtime requis** : GPU T4 (Colab gratuit)

---

In [ ]:
# ══════ CELLULE 1 — INSTALLATION ══════
# WHAM + Torch CUDA + Detectron2 + Google Generative AI
# Duree estimee : 8-12 min sur Colab T4 (1 seule fois par session)

import os, subprocess, sys

# 1. Dependances systeme
print("[1/7] Installation ffmpeg + deps systeme...")
os.system("apt-get install -y ffmpeg libgl1 > /dev/null 2>&1")
print("  OK")

# 2. PyTorch CUDA (T4 = CUDA 11.8)
print("[2/7] Installation PyTorch CUDA 11.8...")
os.system("pip install -q torch==2.1.0 torchvision==0.16.0 --index-url https://download.pytorch.org/whl/cu118")
print("  OK")

# 3. Dependances WHAM tier (mmcv, ViTPose)
print("[3/7] Installation mmcv + mmdet (ViTPose requis par WHAM)...")
os.system("pip install -q openmim")
os.system("mim install -q mmcv==2.0.1 mmdet==3.1.0 mmpose==1.1.0")
print("  OK")

# 4. Detectron2
print("[4/7] Installation Detectron2...")
os.system("pip install -q 'git+https://github.com/facebookresearch/detectron2.git'")
print("  OK")

# 5. Google Generative AI
print("[5/7] Installation google-generativeai...")
os.system("pip install -q google-generativeai")
print("  OK")

# 6. WHAM
print("[6/7] Clonage + installation WHAM...")
WHAM_DIR = os.path.expanduser("~/WHAM")
if not os.path.exists(WHAM_DIR):
    os.system("git clone -q https://github.com/yohanshin/WHAM.git ~/WHAM")
    os.system("pip install -q -r ~/WHAM/requirements.txt")
    print("  WHAM clone et installe")
else:
    print("  WHAM deja present")

# 7. ANIMA-MECHANICUS (codebase)
print("[7/7] Clonage ANIMA-MECHANICUS...")
REPO_DIR = "/content/ANIMA-MECHANICUS"
if os.path.exists(REPO_DIR):
    os.system(f"cd {REPO_DIR} && git pull -q")
    print("  Repo mis a jour")
else:
    os.system(f"git clone -q https://github.com/kioka8877-ux/ANIMA-MECHANICUS.git {REPO_DIR}")
    print("  Repo clone")

# Autres deps Python
os.system("pip install -q numpy scipy opencv-python-headless 'scenedetect[opencv]'")

print("\n[U-ALPHA] Installation terminee.")
print("")
print("IMPORTANT — Modeles SMPL requis :")
print("  1. S'inscrire gratuitement sur : https://smpl.is.tue.mpg.de/")
print("  2. Telecharger : SMPL_python_v.1.1.0.zip")
print("  3. Extraire SMPL_NEUTRAL.pkl")
print("  4. Uploader vers : /content/body_models/smpl/SMPL_NEUTRAL.pkl")
print("     (via le panneau Fichiers de Colab, ou Google Drive monte)")

In [ ]:
# ══════ CELLULE 2 — CONFIGURATION ══════

import os

# @title Parametres U-ALPHA

# Cle API Gemini (gratuite sur https://aistudio.google.com)
GEMINI_API_KEY = ""  # @param {type:"string"}

# FPS cible de l'animation exportee
FPS_CIBLE = 30  # @param [30, 60, 120] {type:"raw"}

# Niveau de lissage Savitzky-Golay
LISSAGE = "moyen"  # @param ["faible", "moyen", "brutal"] {type:"string"}

# Activer la translation globale (root motion)
ROOT_MOTION = True  # @param {type:"boolean"}

# Seuil qualite Gemini — segments sous ce score sont exclus automatiquement
QUALITY_THRESHOLD = 0.6  # @param {type:"number"}

# Chemin modele SMPL (voir instructions Cellule 1)
SMPL_MODEL_PATH = "/content/body_models/smpl/SMPL_NEUTRAL.pkl"  # @param {type:"string"}

# Chemin depot WHAM
WHAM_DIR = os.path.expanduser("~/WHAM")

# Dossier de sortie des .npz
OUTPUT_DIR = "/content/alpha_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- Validation ---
ok = True
if not GEMINI_API_KEY:
    print("[WARN] Cle Gemini API non configuree — obtenir gratuitement sur https://aistudio.google.com")
    ok = False
else:
    print("  OK  Gemini API key configuree")

if not os.path.exists(SMPL_MODEL_PATH):
    print(f"[WARN] Modele SMPL introuvable : {SMPL_MODEL_PATH}")
    print("       Voir instructions Cellule 1 pour l'obtenir")
    ok = False
else:
    print(f"  OK  Modele SMPL : {SMPL_MODEL_PATH}")

if not os.path.exists(WHAM_DIR):
    print(f"[WARN] WHAM introuvable : {WHAM_DIR} — relancer Cellule 1")
    ok = False
else:
    print(f"  OK  WHAM : {WHAM_DIR}")

if ok:
    print(f"\n[CONFIG] FPS={FPS_CIBLE} | Lissage={LISSAGE} | RootMotion={ROOT_MOTION} | Seuil={QUALITY_THRESHOLD}")
    print("[U-ALPHA] Configuration validee — passer a la Cellule 3")

In [ ]:
# ══════ CELLULE 3 — UPLOAD VIDEO ══════
# Uploader la video .mp4 source (humain reel uniquement, max 60 secondes)

from google.colab import files as colab_files
import os

VIDEO_PATH = None

# --- Option A : Upload direct depuis l'ordinateur ---
print("Option A — Upload direct depuis votre ordinateur :")
print("  Selectionnez un fichier .mp4 dans le dialogue ci-dessous")
print("  (appuyez sur Annuler si vous preferez l'Option B)\n")

try:
    uploaded = colab_files.upload()
    for fname, data in uploaded.items():
        if fname.lower().endswith(".mp4"):
            VIDEO_PATH = f"/content/{fname}"
            with open(VIDEO_PATH, "wb") as f:
                f.write(data)
            size_mb = os.path.getsize(VIDEO_PATH) / 1024 / 1024
            print(f"  OK  Video uploadee : {fname} ({size_mb:.1f} MB)")
            break
except Exception:
    pass

if not VIDEO_PATH:
    # --- Option B : Chemin manuel (ex: Google Drive monte) ---
    print("Option B — Chemin manuel (Google Drive ou chemin Colab) :")
    # Remplacer la chaine ci-dessous par le chemin reel de la video
    VIDEO_PATH_MANUEL = ""  # @param {type:"string"}
    if VIDEO_PATH_MANUEL and os.path.exists(VIDEO_PATH_MANUEL):
        VIDEO_PATH = VIDEO_PATH_MANUEL
        size_mb = os.path.getsize(VIDEO_PATH) / 1024 / 1024
        print(f"  OK  Video : {os.path.basename(VIDEO_PATH)} ({size_mb:.1f} MB)")
    else:
        print("  [ERREUR] Aucune video chargee.")
        print("           Uploader une video .mp4 ou configurer VIDEO_PATH_MANUEL")

# Afficher infos video si chargee
if VIDEO_PATH:
    import cv2
    cap = cv2.VideoCapture(VIDEO_PATH)
    src_fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration = total_frames / src_fps if src_fps > 0 else 0
    cap.release()
    print(f"  Duree : {duration:.1f}s | FPS source : {src_fps:.1f} | Frames : {total_frames}")
    if duration > 60:
        print("  [WARN] Duree > 60s — pipeline prevu pour max 60 secondes")

In [ ]:
# ══════ CELLULE 4 — ANALYSE GEMINI 2.0 FLASH ══════
# Identification intelligente des segments exploitables pour WHAM

import sys
sys.path.insert(0, "/content/ANIMA-MECHANICUS/U-ALPHA/codebase")

from motus_extract import analyze_video_gemini, print_gemini_report, filter_segments

assert GEMINI_API_KEY, "[ERREUR] Configurer GEMINI_API_KEY en Cellule 2"
assert VIDEO_PATH and os.path.exists(VIDEO_PATH), "[ERREUR] Charger une video en Cellule 3"

print("[U-ALPHA] Analyse Gemini 2.0 Flash en cours...")
print("  Upload de la video vers les serveurs Google (quelques secondes)...\n")

GEMINI_DATA = analyze_video_gemini(VIDEO_PATH, GEMINI_API_KEY)
print_gemini_report(GEMINI_DATA)

# Filtrage automatique selon QUALITY_THRESHOLD
ALL_SEGMENTS = filter_segments(GEMINI_DATA, QUALITY_THRESHOLD)

print(f"\n{'='*60}")
print(f"[FILTRE] {len(ALL_SEGMENTS)} segment(s) retenus (seuil qualite >= {QUALITY_THRESHOLD}) :")
for s in ALL_SEGMENTS:
    duration_seg = s['end_s'] - s['start_s']
    print(f"  P{s['person_id']} | {s['start_s']:.1f}s - {s['end_s']:.1f}s "
          f"({duration_seg:.1f}s) | {s['corps_visible']} | "
          f"qualite={s['qualite_estimee']:.2f} | {s['type_mouvement']}")
print(f"{'='*60}")

# Avertissements camera
cam = GEMINI_DATA.get("camera", {})
if cam.get("mouvement") == "agitee":
    print("\n[WARN] Camera agitee detectee — root motion peut etre peu fiable")
if cam.get("zoom_detecte"):
    print("[WARN] Zoom detecte — root motion peut etre fausse")

In [ ]:
# ══════ CELLULE 5 — VALIDATION DES SEGMENTS ══════
# Par defaut : tous les segments retenus par Gemini sont traites
# Modifier SEGMENTS_TO_PROCESS si vous voulez affiner la selection

# --- Selection par defaut (tous les segments valides) ---
SEGMENTS_TO_PROCESS = ALL_SEGMENTS

# === Ajustements manuels optionnels ===
# Decommenter et modifier selon votre cas :

# Traiter uniquement la personne 0
# SEGMENTS_TO_PROCESS = [s for s in ALL_SEGMENTS if s['person_id'] == 0]

# Traiter uniquement les segments apres 5 secondes
# SEGMENTS_TO_PROCESS = [s for s in ALL_SEGMENTS if s['start_s'] >= 5.0]

# Traiter uniquement les segments 'complet' (exclure les partiels)
# SEGMENTS_TO_PROCESS = [s for s in ALL_SEGMENTS if s['corps_visible'] == 'complet']

# Traiter un segment specifique manuellement (remplacer les valeurs)
# SEGMENTS_TO_PROCESS = [{'person_id': 0, 'start_s': 2.5, 'end_s': 38.0,
#     'corps_visible': 'complet', 'qualite_estimee': 0.9, 'type_mouvement': 'danse'}]

# === Affichage de la selection finale ===
print(f"[U-ALPHA] {len(SEGMENTS_TO_PROCESS)} segment(s) valides pour WHAM :")
total_dur = sum(s['end_s'] - s['start_s'] for s in SEGMENTS_TO_PROCESS)
for s in SEGMENTS_TO_PROCESS:
    dur = s['end_s'] - s['start_s']
    n_frames_est = int(dur * FPS_CIBLE)
    print(f"  P{s['person_id']} | {s['start_s']:.1f}s - {s['end_s']:.1f}s "
          f"({dur:.1f}s, ~{n_frames_est} frames @ {FPS_CIBLE} FPS) | "
          f"{s['corps_visible']}")

print(f"\n  Duree totale a traiter : {total_dur:.1f}s")

if not SEGMENTS_TO_PROCESS:
    print("[ERREUR] Aucun segment selectionne — relancer la Cellule 4 ou ajuster le seuil")
else:
    print("\n[U-ALPHA] Validation OK — passer a la Cellule 6 pour lancer WHAM")

In [ ]:
# ══════ CELLULE 6 — EXTRACTION WHAM → SMPL→R15 → .npz ══════
# Duree estimee : 1-3 min par segment de 30s sur Colab T4

import sys, os, tempfile
import numpy as np
import cv2

sys.path.insert(0, "/content/ANIMA-MECHANICUS/U-ALPHA/codebase")
from motus_extract import (
    run_wham, smpl_to_r15, fill_gaps,
    smooth_array, smooth_extremities,
    temporal_resample, normalize_quats,
    export_npz, SMOOTH_PRESETS
)

assert SEGMENTS_TO_PROCESS, "[ERREUR] Valider les segments en Cellule 5"

# Infos video source
cap = cv2.VideoCapture(VIDEO_PATH)
src_fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
duration = total_frames / src_fps
cap.release()
print(f"[U-ALPHA] Source : {src_fps:.1f} FPS, {duration:.1f}s, {total_frames} frames")

# ── Etape 1 : WHAM ────────────────────────────────────────────────────────
print("\n[U-ALPHA] Etape 1/3 — Extraction WHAM (poses SMPL)...")
with tempfile.TemporaryDirectory() as tmp_dir:
    wham_tracks = run_wham(VIDEO_PATH, SEGMENTS_TO_PROCESS, WHAM_DIR, tmp_dir)

if not wham_tracks:
    raise RuntimeError("[ERREUR] WHAM n'a produit aucun resultat. Verifier l'installation et les modeles SMPL.")

print(f"  {len(wham_tracks)} piste(s) extraite(s) par WHAM")

# ── Etapes 2-3 : Retargeting + Lissage + Export ───────────────────────────
print("\n[U-ALPHA] Etape 2/3 — Retargeting SMPL→R15 + Lissage...")
win, poly = SMOOTH_PRESETS[LISSAGE]
no_root_motion = not ROOT_MOTION
n_persons = len(wham_tracks)
NPZ_FILES = []

for pid in sorted(wham_tracks.keys()):
    track = wham_tracks[pid]
    print(f"  Personne {pid} : {len(track['poses'])} frames brutes")

    # Interpolation des trous d'occlusion
    fill_gaps(track["poses"], track["transl"])

    indices = sorted(track["poses"].keys())
    poses_aa = np.array([track["poses"][i] for i in indices])  # (N, 24, 3)
    transl   = np.array([track["transl"][i] for i in indices]) # (N, 3)

    # Retargeting SMPL → R15
    rots, root_pos = smpl_to_r15(poses_aa, transl)
    if no_root_motion:
        root_pos = np.zeros_like(root_pos)

    # Lissage global
    root_pos = smooth_array(root_pos, win, poly)
    rots = smooth_array(rots.reshape(len(rots), -1), win, poly).reshape(-1, 15, 4)
    rots = normalize_quats(rots)

    # Lissage renforce extremites (mains, pieds)
    rots = smooth_extremities(rots, max(win * 2 + 1, 15))

    # Resampling vers FPS cible
    root_pos = temporal_resample(root_pos, src_fps, FPS_CIBLE)
    rots = temporal_resample(
        rots.reshape(len(rots), -1), src_fps, FPS_CIBLE
    ).reshape(-1, 15, 4)
    rots = normalize_quats(rots)

    # Export .npz
    out_path = f"{OUTPUT_DIR}/motus_core_P{pid}.npz"
    export_npz(rots, root_pos, FPS_CIBLE, src_fps, duration, pid, n_persons, out_path)
    NPZ_FILES.append(out_path)
    size_kb = os.path.getsize(out_path) / 1024
    print(f"  → {os.path.basename(out_path)} | {rots.shape[0]} frames @ {FPS_CIBLE} FPS | {size_kb:.1f} KB")

print(f"\n[U-ALPHA] {len(NPZ_FILES)} fichier(s) .npz exportes dans {OUTPUT_DIR}")
print("[U-ALPHA] Passer a la Cellule 7 pour telecharger les .npz")

In [ ]:
# ══════ CELLULE 7 — TELECHARGEMENT DES .npz ══════
# Ces fichiers sont l'input de la Fregate U-GAMMA (La Forge)

from google.colab import files as colab_files
import os

assert NPZ_FILES, "[ERREUR] Aucun .npz genere — relancer la Cellule 6"

print(f"[U-ALPHA] {len(NPZ_FILES)} fichier(s) .npz prets au telechargement :")
for npz_path in NPZ_FILES:
    size_kb = os.path.getsize(npz_path) / 1024
    print(f"  {os.path.basename(npz_path)} ({size_kb:.1f} KB)")

print("\nTelechargement en cours...")
for npz_path in NPZ_FILES:
    colab_files.download(npz_path)

print("\n[U-ALPHA] Mission accomplie — Fregate U-ALPHA terminee.")
print("Prochain pas : Ouvrir ANIMA_MECHANICUS_GAMMA.ipynb")
print("  → Uploader ces .npz dans la Fregate U-GAMMA (La Forge)")
print("  → Obtenir les .fbx prets pour Roblox Studio")